# 15 — Custom Fine-Tuning: Domain-Specific NLI Models

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anulum/director-ai/blob/main/notebooks/15_custom_fine_tuning.ipynb)

Train a domain-specific NLI model on your own data to dramatically
improve accuracy for your use case. The fine-tuning pipeline includes
data validation, anti-forgetting safeguards, regression testing, and
ONNX export.

**What you'll learn:**
1. Prepare training data (JSONL format)
2. Validate data before training
3. Fine-tune with anti-forgetting protection
4. Benchmark against baselines
5. Export to ONNX for production
6. Fine-tuning REST API for self-service

Requires `pip install director-ai[finetune]` (torch, transformers, datasets).

In [ ]:
!pip install -q director-ai[finetune]

## 1. Training Data Format

Each line is a JSON object with `premise`, `hypothesis`, and `label`:

```jsonl
{"premise": "Refunds available within 30 days.", "hypothesis": "I can get a refund after 30 days.", "label": 0}
{"premise": "Refunds available within 30 days.", "hypothesis": "Returns accepted in the first month.", "label": 1}
```

Labels:
- `1` = entailment (hypothesis follows from premise)
- `0` = contradiction (hypothesis contradicts premise)

In [ ]:
import json
import tempfile
from pathlib import Path

# Generate synthetic training data
training_data = [
    # Entailments (label=1)
    {
        "premise": "Maximum dosage is 400mg every 6 hours.",
        "hypothesis": "Take no more than 400mg at a time.",
        "label": 1,
    },
    {
        "premise": "The API rate limit is 1000 requests per minute.",
        "hypothesis": "You can make up to 1000 API calls each minute.",
        "label": 1,
    },
    {
        "premise": "Free shipping on orders over $50.",
        "hypothesis": "Orders above fifty dollars ship for free.",
        "label": 1,
    },
    {
        "premise": "Data is encrypted at rest using AES-256.",
        "hypothesis": "Stored data is protected with AES-256 encryption.",
        "label": 1,
    },
    {
        "premise": "Support available Monday-Friday 9am-5pm EST.",
        "hypothesis": "You can reach support on weekdays during business hours.",
        "label": 1,
    },
    # Contradictions (label=0)
    {
        "premise": "Maximum dosage is 400mg every 6 hours.",
        "hypothesis": "You can safely take 800mg every 4 hours.",
        "label": 0,
    },
    {
        "premise": "The API rate limit is 1000 requests per minute.",
        "hypothesis": "There is no rate limit on API usage.",
        "label": 0,
    },
    {
        "premise": "Free shipping on orders over $50.",
        "hypothesis": "All orders ship for free regardless of amount.",
        "label": 0,
    },
    {
        "premise": "Data is encrypted at rest using AES-256.",
        "hypothesis": "Data is stored in plain text.",
        "label": 0,
    },
    {
        "premise": "Support available Monday-Friday 9am-5pm EST.",
        "hypothesis": "24/7 support is available including weekends.",
        "label": 0,
    },
]

# Write to JSONL
data_dir = Path(tempfile.mkdtemp())
data_path = data_dir / "training_data.jsonl"

with open(data_path, "w", encoding="utf-8") as f:
    for row in training_data:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print(f"Wrote {len(training_data)} samples to {data_path}")
print(f"Entailments: {sum(1 for r in training_data if r['label'] == 1)}")
print(f"Contradictions: {sum(1 for r in training_data if r['label'] == 0)}")

## 2. Data Validation

Always validate before training. The validator checks format,
class balance, duplicates, and estimates training time + cost.

In [ ]:
from director_ai.core import validate_finetune_data

report = validate_finetune_data(str(data_path), epochs=3)

print(f"Valid: {report.is_valid}")
print(f"Samples: {report.total_samples}")
print(f"Labels: {report.label_distribution}")
print(f"Class balance: {report.class_balance_ratio:.2f}")
print(f"Duplicates: {report.duplicate_count}")
print(f"Est. train time: {report.estimated_train_time_min:.1f} min")
print(f"Est. cost (A100): ${report.estimated_cost_usd:.2f}")

if report.warnings:
    print("\nWarnings:")
    for w in report.warnings:
        print(f"  ⚠ {w}")

if report.errors:
    print("\nErrors:")
    for e in report.errors:
        print(f"  ✗ {e}")

## 3. Fine-Tuning Configuration

The `FinetuneConfig` controls all training parameters.

In [ ]:
from director_ai.core import FinetuneConfig

config = FinetuneConfig(
    output_dir=str(data_dir / "model_output"),
    epochs=3,
    batch_size=16,
    learning_rate=2e-5,
    # Anti-forgetting: mix 20% general NLI data (MNLI/SNLI)
    mix_general_data=True,
    general_data_ratio=0.2,
    # Early stopping: stop if no improvement for 3 evaluations
    early_stopping_patience=3,
    # Class weighting: handle imbalanced datasets
    class_weighted_loss=True,
    # Auto-benchmark against general NLI after training
    auto_benchmark=True,
    # Export to ONNX after training
    auto_onnx_export=False,
)

print("Fine-tuning configuration:")
for field_name in [
    "output_dir",
    "epochs",
    "batch_size",
    "learning_rate",
    "mix_general_data",
    "general_data_ratio",
    "early_stopping_patience",
    "class_weighted_loss",
]:
    print(f"  {field_name}: {getattr(config, field_name)}")

## 4. Training (GPU Required)

```python
from director_ai.core import finetune_nli

result = finetune_nli(
    train_path=str(data_path),
    eval_path=str(eval_data_path),  # optional
    config=config,
)

print(f"Best balanced accuracy: {result.best_balanced_accuracy:.1%}")
print(f"Model saved to: {result.output_dir}")
print(f"Eval metrics: {result.eval_metrics}")
print(f"Regression report: {result.regression_report}")
```

Training takes ~15 minutes per epoch on an A100 for 10K samples.

## 5. Anti-Forgetting Protection

Fine-tuning on domain data risks **catastrophic forgetting** — the
model loses general NLI ability while learning domain specifics.

Director-AI mitigates this with:

| Mechanism | How |
|-----------|-----|
| General data mixing | 20% of each batch is general NLI (MNLI/SNLI) |
| Auto-benchmark | Tests against MNLI/FEVER/VitaminC after training |
| Regression gate | Fails if general accuracy drops >5% from baseline |
| Class weighting | Upweights minority class to prevent bias |

The regression report shows general-benchmark performance:

```json
{
  "mnli_accuracy": 0.891,
  "mnli_baseline": 0.903,
  "mnli_delta": -0.012,
  "regression_detected": false
}
```

## 6. Benchmarking a Fine-Tuned Model

```python
from director_ai.core import benchmark_finetuned_model

report = benchmark_finetuned_model(
    model_path="./model_output",
    test_data_path="./test_data.jsonl",
    baseline_model="yaxili96/FactCG-DeBERTa-v3-Large",
)

print(f"Domain accuracy: {report.domain_accuracy:.1%}")
print(f"General accuracy: {report.general_accuracy:.1%}")
print(f"Regression: {report.regression_detected}")
print(f"Recommendation: {report.recommendation}")
```

## 7. ONNX Export for Production

Export fine-tuned model to ONNX for 3-5x faster inference.

```python
from director_ai.core.nli import export_onnx

export_onnx(
    model_name="./model_output",
    output_dir="./model_onnx",
    quantize="int8",  # optional: int8 or fp16
)

# Use in production
from director_ai.core.config import DirectorConfig

config = DirectorConfig(
    scorer_backend="onnx",
    onnx_path="./model_onnx",
)
```

## 8. Fine-Tuning REST API

Enable self-service fine-tuning via the REST API:

```bash
# Upload and validate data
curl -X POST http://localhost:8080/v1/finetune/validate \
  -F 'file=@training_data.jsonl'

# Start training job
curl -X POST http://localhost:8080/v1/finetune/start \
  -F 'file=@training_data.jsonl' \
  -F 'epochs=3' \
  -F 'mix_general_data=true'

# Check progress
curl http://localhost:8080/v1/finetune/{job_id}

# Get results
curl http://localhost:8080/v1/finetune/{job_id}/result

# Activate model
curl -X POST http://localhost:8080/v1/finetune/{job_id}/activate

# Rollback
curl -X POST http://localhost:8080/v1/finetune/{job_id}/rollback
```

## Data Preparation Tips

| Tip | Why |
|-----|-----|
| 500+ samples minimum | Below this, fine-tuning rarely helps |
| Balanced classes | Equal entailment/contradiction samples |
| Domain-specific language | Use your actual product terms |
| Hard negatives | Include plausible-but-wrong hypotheses |
| Diverse premises | Cover all your knowledge base topics |
| Real user queries | Mine actual user questions as hypotheses |

In [ ]:
# Cleanup
import shutil

shutil.rmtree(data_dir, ignore_errors=True)
print("Cleaned up temporary files.")